In [ ]:
# Canonical analysis definitions (single source of truth: analysis/_canonical_defs.py).
# Importing these overrides any local fallback definitions later in the notebook.
# Edits to filters / score-mapping / bucket-rule belong in _canonical_defs.py, not here.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("analysis").resolve()))
from _canonical_defs import (
    ANCHOR, L2_CELLS, AXIS_CELLS, VERDICT_SCORE,
    candidate_score, sign,
    any_tier_disagreement, boundary_flip, strict_reversal,
    same_side_magnitude_shift, strong_to_weak_shift,
    assign_bucket,
)


# 11 — Inter-judge agreement (Cohen's linear-weighted κ)

Pairwise chance-corrected agreement statistics between the three judges on the anchor cell (`C1F1D1`):

1. GPT-4.1 ↔ Claude Sonnet 4.5
2. GPT-4.1 ↔ Gemini 2.5 Pro
3. Claude Sonnet 4.5 ↔ Gemini 2.5 Pro

Linear-weighted Cohen's κ is used because the verdict scale is ordinal (5-point: `A>>B` to `B>>A`). Values: 0 = chance agreement; 1 = perfect agreement; negative = worse than chance.

**Prerequisite**: `10_analysis.ipynb` has been re-run with the Gemini data loaded, so `results/verdict_scores_wide.csv` contains columns `C1F1D1` (GPT-4.1 anchor), `L1_claude`, and `L1_gemini`.

**Outputs**: `results/inter_judge_agreement.csv` — 3 rows × {pair, n_paired, kappa_linear, ci_lo, ci_hi}.

**Paper use**:
- §4.5 Robustness: one sentence reporting the three κs.
- New appendix `EMNLP/appendices/M_inter_judge_agreement.{csv,md}`.
- Limitations: remove the existing 'no inter-judge κ' bullet.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

## Load wide-format verdict scores

In [2]:
RESULTS_DIR = Path('results')
wide = pd.read_csv(RESULTS_DIR / 'verdict_scores_wide.csv', index_col=0)
print(f'wide shape: {wide.shape}')
print(f'columns: {list(wide.columns)}')

required = ['C1F1D1', 'L1_claude', 'L1_gemini']
missing = [c for c in required if c not in wide.columns]
if missing:
    raise RuntimeError(
        f'Missing columns {missing} in verdict_scores_wide.csv. '
        f'Re-run 10_analysis.ipynb after 05b_L1_swap_gemini_flash.ipynb completes.'
    )

wide shape: (500, 10)
columns: ['C1F1D1', 'C1F1D2', 'C1F2D1', 'C1F2D2', 'C2F1D1', 'C2F1D2', 'C2F2D1', 'C2F2D2', 'L1_claude', 'L1_gemini']


## Map mean scores back to 5-tier ordinal verdicts

Each cell's `mean_score` is the average of two-round candidate-relative scores (range −2 to +2). Convert back to the original 5-point tier by binning.

In [3]:
def score_to_tier(s):
    """Map mean candidate-score to 5-point ordinal verdict (-2..+2)."""
    if pd.isna(s):
        return np.nan
    if s < -1.5:
        return -2  # A>>B (baseline strongly wins)
    if s < -0.5:
        return -1  # A>B
    if s <= 0.5:
        return 0   # A=B
    if s <= 1.5:
        return 1   # B>A (candidate wins)
    return 2       # B>>A

tiers = wide[['C1F1D1', 'L1_claude', 'L1_gemini']].apply(lambda col: col.map(score_to_tier))
tiers = tiers.rename(columns={'C1F1D1': 'gpt4', 'L1_claude': 'claude', 'L1_gemini': 'gemini'})
print('Tier counts per judge:')
for col in tiers.columns:
    print(f'  {col}: {tiers[col].value_counts().sort_index().to_dict()}')

Tier counts per judge:
  gpt4: {-2.0: 33, -1.0: 99, 0.0: 178, 1.0: 158, 2.0: 28}
  claude: {-2.0: 39, -1.0: 115, 0.0: 191, 1.0: 140, 2.0: 13}
  gemini: {-2.0: 62, -1.0: 66, 0.0: 172, 1.0: 129, 2.0: 56}


## Pairwise linear-weighted Cohen's κ with bootstrap CI

In [4]:
def kappa_with_ci(a, b, n_boot=1000, seed=42):
    """Linear-weighted Cohen's κ with bootstrap 95% CI."""
    df = pd.DataFrame({'a': a, 'b': b}).dropna()
    n = len(df)
    if n < 2:
        return None
    point = cohen_kappa_score(df['a'].astype(int), df['b'].astype(int), weights='linear')
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_boot):
        idx = rng.choice(df.index, size=n, replace=True)
        sub = df.loc[idx]
        try:
            k = cohen_kappa_score(sub['a'].astype(int), sub['b'].astype(int), weights='linear')
            samples.append(k)
        except Exception:
            continue
    lo, hi = np.percentile(samples, [2.5, 97.5])
    return {'n': n, 'kappa': point, 'ci_lo': lo, 'ci_hi': hi, 'n_boot': len(samples)}

pairs = [
    ('gpt4_vs_claude', 'gpt4', 'claude'),
    ('gpt4_vs_gemini', 'gpt4', 'gemini'),
    ('claude_vs_gemini', 'claude', 'gemini'),
]

rows = []
for name, a_col, b_col in pairs:
    res = kappa_with_ci(tiers[a_col], tiers[b_col])
    rows.append({
        'pair': name,
        'judge_a': a_col,
        'judge_b': b_col,
        'n_paired': res['n'],
        'kappa_linear': round(res['kappa'], 4),
        'ci_lo': round(res['ci_lo'], 4),
        'ci_hi': round(res['ci_hi'], 4),
    })
    print(f"  {name}: n={res['n']}, κ={res['kappa']:.3f}, 95% CI [{res['ci_lo']:.3f}, {res['ci_hi']:.3f}]")

kappa_df = pd.DataFrame(rows)
kappa_df

  gpt4_vs_claude: n=494, κ=0.549, 95% CI [0.495, 0.602]
  gpt4_vs_gemini: n=481, κ=0.435, 95% CI [0.378, 0.492]
  claude_vs_gemini: n=483, κ=0.478, 95% CI [0.425, 0.530]


,pair,judge_a,judge_b,n_paired,kappa_linear,ci_lo,ci_hi
0,gpt4_vs_claude,gpt4,claude,494,0.5493,0.4950,0.6018
1,gpt4_vs_gemini,gpt4,gemini,481,0.4348,0.3781,0.4919
2,claude_vs_gemini,claude,gemini,483,0.4778,0.4247,0.5301


## Interpretation reference

Landis & Koch (1977) reference scale for κ — informal but widely cited:

| κ | Strength of agreement |
|---|---|
| < 0.00 | worse than chance |
| 0.00 – 0.20 | slight |
| 0.21 – 0.40 | fair |
| 0.41 – 0.60 | moderate |
| 0.61 – 0.80 | substantial |
| 0.81 – 1.00 | almost perfect |

Headline read after computation: report all three κs in the paper; if any CI excludes the others, that's evidence of judge-pair-specific divergence beyond a uniform 'judges disagree' story.

## Save

In [5]:
out_path = RESULTS_DIR / 'inter_judge_agreement.csv'
kappa_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

# Append to metrics_summary.json if it exists
metrics_path = RESULTS_DIR / 'metrics_summary.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    metrics['inter_judge_agreement'] = {
        row['pair']: {
            'kappa_linear': row['kappa_linear'],
            'ci_lo': row['ci_lo'],
            'ci_hi': row['ci_hi'],
            'n_paired': int(row['n_paired']),
        }
        for _, row in kappa_df.iterrows()
    }
    metrics_path.write_text(json.dumps(metrics, indent=2))
    print(f'Updated: {metrics_path}')

print('\nDone. Next: feed numbers into EMNLP/appendices/M_inter_judge_agreement.{csv,md}')

Saved: results/inter_judge_agreement.csv
Updated: results/metrics_summary.json

Done. Next: feed numbers into EMNLP/appendices/M_inter_judge_agreement.{csv,md}
